# Advanced Machine Learning (21CSE425T) — Unit 3: Ensemble Learning Models

Standalone Colab notebook for **T7–T9**: AdaBoost/Gradient Boosting, Bagging + bias–variance, and ensemble-based Face Recognition. Run top-to-bottom; the setup cell installs all dependencies.

---
# Unit 3 — Ensemble Learning Models

Installs scikit-learn, OpenCV, pandas, mlxtend, then covers **Boosting** (T7), **Bagging & the
bias–variance trade-off** (T8) and **ensemble-based face recognition** (T9).

### Unit 3 — Setup & installs

In [ ]:
!pip -q install scikit-learn pandas opencv-python mlxtend 2>/dev/null
import sklearn, pandas as pd, numpy as np
import cv2
print("scikit-learn:", sklearn.__version__)
print("pandas      :", pd.__version__)
print("OpenCV      :", cv2.__version__)

## T7 — AdaBoost & Gradient Boosting

Two boosting families on the **Breast Cancer Wisconsin** dataset: **AdaBoost** (re-weights misclassified
samples) and **Gradient Boosting** (fits each tree to the previous ensemble's residuals). We compare
accuracy, ROC-AUC and the most informative features.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import numpy as np, matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)
feat_names = load_breast_cancer().feature_names
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

models = {
    "Decision stump (baseline)": DecisionTreeClassifier(max_depth=1, random_state=0),
    "AdaBoost (200 stumps)":     AdaBoostClassifier(
                                    estimator=DecisionTreeClassifier(max_depth=1),
                                    n_estimators=200, learning_rate=0.5, random_state=0),
    "Gradient Boosting":         GradientBoostingClassifier(
                                    n_estimators=200, learning_rate=0.1,
                                    max_depth=3, random_state=0),
}
for name, clf in models.items():
    clf.fit(Xtr, ytr)
    proba = clf.predict_proba(Xte)[:, 1]
    print(f"{name:28s} | acc={accuracy_score(yte, clf.predict(Xte)):.3f} "
          f"| AUC={roc_auc_score(yte, proba):.3f}")

# ---- Top features from the Gradient Boosting model ----
gb = models["Gradient Boosting"]
idx = np.argsort(gb.feature_importances_)[::-1][:10]
plt.figure(figsize=(8,4))
plt.barh(range(len(idx)), gb.feature_importances_[idx][::-1])
plt.yticks(range(len(idx)), [feat_names[i] for i in idx[::-1]], fontsize=8)
plt.xlabel("Importance"); plt.title("T7 — Gradient Boosting: top-10 features")
plt.tight_layout(); plt.show()

## T8 — Bagging & the bias–variance trade-off

**Bagging** (bootstrap aggregating) trains many high-variance learners on resampled data and averages
them. Using `mlxtend`'s `bias_variance_decomp`, we show that bagging a deep decision tree leaves **bias**
roughly unchanged while sharply cutting **variance** — exactly the mechanism behind Random Forests.

In [ ]:
from mlxtend.evaluate import bias_variance_decomp
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

single = DecisionTreeClassifier(random_state=0)                      # high variance
bagged = BaggingClassifier(DecisionTreeClassifier(random_state=0),
                           n_estimators=100, random_state=0)         # variance-reduced

print(f"{'Model':22s}{'avg loss':>12}{'bias^2':>12}{'variance':>12}")
for name, model in [("Single Decision Tree", single), ("Bagging (100 trees)", bagged)]:
    loss, bias, var = bias_variance_decomp(
        model, Xtr, ytr, Xte, yte,
        loss="0-1_loss", num_rounds=100, random_seed=0)
    print(f"{name:22s}{loss:12.4f}{bias:12.4f}{var:12.4f}")

print("\nNote how variance drops markedly under bagging while bias stays similar.")

## T9 — Face recognition with ensemble techniques

On the **Olivetti faces** dataset (40 people × 10 images), we reduce dimensionality with **PCA
(eigenfaces)**, then classify identities with three ensembles — **Random Forest**, **Bagging(SVM)** and
**AdaBoost** — and visualise predictions.

In [ ]:
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
import numpy as np, matplotlib.pyplot as plt

faces = fetch_olivetti_faces(shuffle=True, random_state=42)   # downloads ~once (small)
X, y = faces.data, faces.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(f"Images: {X.shape[0]} of size 64x64 | identities: {len(np.unique(y))}")

pca = PCA(n_components=100, whiten=True, random_state=42).fit(Xtr)   # eigenfaces

ensembles = {
    "Random Forest":  RandomForestClassifier(n_estimators=300, random_state=0),
    "Bagging(SVM)":   BaggingClassifier(SVC(kernel="rbf", C=10, gamma="scale"),
                                        n_estimators=15, random_state=0),
    "AdaBoost":       AdaBoostClassifier(n_estimators=200, random_state=0),
}
fitted = {}
for name, clf in ensembles.items():
    pipe = make_pipeline(pca, clf).fit(Xtr, ytr)   # PCA -> ensemble
    acc = accuracy_score(yte, pipe.predict(Xte))
    fitted[name] = pipe
    print(f"{name:14s} accuracy: {acc:.3f}")

In [ ]:
# ---- Show a few predictions from the best ensemble ----
best_name = max(fitted, key=lambda n: accuracy_score(yte, fitted[n].predict(Xte)))
best = fitted[best_name]
preds = best.predict(Xte)

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, img, p, t in zip(axes.ravel(), Xte, preds, yte):
    ax.imshow(img.reshape(64, 64), cmap="gray")
    ok = (p == t)
    ax.set_title(f"pred {p}\n{'OK' if ok else 'true '+str(t)}",
                 color=("green" if ok else "red"), fontsize=9)
    ax.axis("off")
fig.suptitle(f"T9 — Face recognition via {best_name} (PCA eigenfaces + ensemble)")
plt.tight_layout(); plt.show()